In [ ]:
# =============================================================================
# 13_compare_image_quality_across_datasets.ipynb
#
# PURPOSE
# =============================================================================
#
# Compare image quality across the three temporal datasets:
#
#   1. DAILY / acquisition-level imagery       -> Notebook 10
#   2. WEEKLY / 7-day composites               -> Notebook 11
#   3. BIWEEKLY / 14-day composites            -> Notebook 12
#
#
# SAME SAMPLE
# =============================================================================
#
# All comparisons are restricted to the sample selected by Notebook 10:
#
#   - 10 treatment sites
#   - 5 counterfactual sites per treatment
#   - 50 counterfactual sites
#   - 60 total spatial units
#
#
# Source:
#
#   finals/daily_datasets/selected_site_sample.csv
#
#
# SENSORS
# =============================================================================
#
#   - Sentinel-1
#   - Sentinel-2
#
#
# GROUPS
# =============================================================================
#
#   - treatment
#   - counterfactual
#
#
# PERIODS
# =============================================================================
#
#   BEFORE:
#       2024-05-10 through 2024-09-26
#
#   AFTER:
#       2024-09-27 through 2025-02-13
#
#
# PRIMARY IMAGE QUALITY DEFINITION
# =============================================================================
#
# A spatial pixel is VALID when AT LEAST ONE output band has a finite value.
#
# Preferred variable:
#
#   valid_pixel_fraction
#
# Fallback:
#
#   valid_pixel_fraction_any_band
#
# Legacy fallback:
#
#   valid_pixel_fraction_all_bands
#
#
# IMPORTANT DISTINCTION
# =============================================================================
#
# IMAGE QUALITY:
#
#   Calculated only for an image that actually exists.
#
#
# TEMPORAL AVAILABILITY:
#
#   Measures whether a scheduled weekly/biweekly period has an image.
#
#
# Missing weekly/biweekly periods are NOT treated as 0%-quality images.
#
#
# MAIN ANALYSIS DIMENSIONS
# =============================================================================
#
# 1. PERIOD DIMENSION
#
# Within each temporal period:
#
#   dataset × sensor × group × period
#
# compare image quality across sites.
#
#
# 2. SITE DIMENSION
#
# For each site:
#
# compare quality across time.
#
#
# 3. OVERALL COMPARISON
#
# Compare:
#
#   daily vs weekly vs biweekly
#
# using:
#
#   mean
#   median
#   minimum
#   maximum
#   standard deviation
#   % >=40%
#   % >=50%
#   % >=60%
#   % >=80%
#   % >=90%
#
#
# 4. TEMPORAL AVAILABILITY
#
# Weekly and biweekly only:
#
#   expected periods
#   available periods
#   missing periods
#   availability rate
#
#
# OUTPUT STRUCTURE
# =============================================================================
#
# finals/
# └── quality/
#
#     ├── 01_daily_image_quality.xlsx
#     ├── 02_weekly_image_quality.xlsx
#     ├── 03_biweekly_image_quality.xlsx
#     ├── 04_image_quality_comparison.xlsx
#     │
#     ├── 01_period_dimension.csv
#     ├── 02_site_dimension.csv
#     ├── 03_overall_comparison.csv
#     ├── 04_quality_distribution.csv
#     ├── 05_temporal_availability.csv
#     ├── 06_S2_treatment_before_matrix.csv
#     ├── 07_S2_treatment_after_matrix.csv
#     ├── 08_S2_counterfactual_before_matrix.csv
#     ├── 09_S2_counterfactual_after_matrix.csv
#     ├── 10_dataset_inventory.csv
#     └── 11_sample_validation.csv
#
# =============================================================================


# =============================================================================
# 1. Packages
# =============================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd


warnings.filterwarnings(
    "ignore"
)


print(
    "Packages loaded successfully."
)


# =============================================================================
# 2. Project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)


FINALS_DIR = (
    BASE_DIR /
    "finals"
)


DAILY_DIR = (
    FINALS_DIR /
    "daily_datasets"
)


WEEKLY_DIR = (
    FINALS_DIR /
    "weekly_datasets"
)


BIWEEKLY_DIR = (
    FINALS_DIR /
    "biweekly"
)


QUALITY_DIR = (
    FINALS_DIR /
    "quality"
)


QUALITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "\nQuality output directory:"
)


print(
    QUALITY_DIR
)


# =============================================================================
# 3. Source files
# =============================================================================

SELECTED_SAMPLE_FILE = (
    DAILY_DIR /
    "selected_site_sample.csv"
)


DAILY_FILE = (
    DAILY_DIR /
    "daily_satellite_inventory.csv"
)


WEEKLY_FILE = (
    WEEKLY_DIR /
    "weekly_image_quality.csv"
)


BIWEEKLY_FILE = (
    BIWEEKLY_DIR /
    "biweekly_image_quality.csv"
)


# =============================================================================
# 4. CSV outputs
# =============================================================================

PERIOD_OUTPUT = (
    QUALITY_DIR /
    "01_period_dimension.csv"
)


SITE_OUTPUT = (
    QUALITY_DIR /
    "02_site_dimension.csv"
)


OVERALL_OUTPUT = (
    QUALITY_DIR /
    "03_overall_comparison.csv"
)


DISTRIBUTION_OUTPUT = (
    QUALITY_DIR /
    "04_quality_distribution.csv"
)


AVAILABILITY_OUTPUT = (
    QUALITY_DIR /
    "05_temporal_availability.csv"
)


S2_TREATMENT_BEFORE_MATRIX_OUTPUT = (
    QUALITY_DIR /
    "06_S2_treatment_before_matrix.csv"
)


S2_TREATMENT_AFTER_MATRIX_OUTPUT = (
    QUALITY_DIR /
    "07_S2_treatment_after_matrix.csv"
)


S2_COUNTERFACTUAL_BEFORE_MATRIX_OUTPUT = (
    QUALITY_DIR /
    "08_S2_counterfactual_before_matrix.csv"
)


S2_COUNTERFACTUAL_AFTER_MATRIX_OUTPUT = (
    QUALITY_DIR /
    "09_S2_counterfactual_after_matrix.csv"
)


DATASET_INVENTORY_OUTPUT = (
    QUALITY_DIR /
    "10_dataset_inventory.csv"
)


SAMPLE_VALIDATION_OUTPUT = (
    QUALITY_DIR /
    "11_sample_validation.csv"
)


# =============================================================================
# 5. Excel outputs
# =============================================================================

DAILY_EXCEL_OUTPUT = (
    QUALITY_DIR /
    "01_daily_image_quality.xlsx"
)


WEEKLY_EXCEL_OUTPUT = (
    QUALITY_DIR /
    "02_weekly_image_quality.xlsx"
)


BIWEEKLY_EXCEL_OUTPUT = (
    QUALITY_DIR /
    "03_biweekly_image_quality.xlsx"
)


COMPARISON_EXCEL_OUTPUT = (
    QUALITY_DIR /
    "04_image_quality_comparison.xlsx"
)


# =============================================================================
# 6. Study design
# =============================================================================

HELENE_REFERENCE_DATE = pd.Timestamp(
    "2024-09-27"
)


STUDY_START = pd.Timestamp(
    "2024-05-10"
)


STUDY_END = pd.Timestamp(
    "2025-02-13"
)


EXPECTED_TREATMENT_SITES = 10


EXPECTED_CONTROLS_PER_TREATMENT = 5


EXPECTED_COUNTERFACTUAL_SITES = (
    EXPECTED_TREATMENT_SITES
    *
    EXPECTED_CONTROLS_PER_TREATMENT
)


EXPECTED_TOTAL_SITES = (
    EXPECTED_TREATMENT_SITES
    +
    EXPECTED_COUNTERFACTUAL_SITES
)


# =============================================================================
# 7. Quality thresholds
# =============================================================================

QUALITY_THRESHOLDS = [

    0.40,

    0.50,

    0.60,

    0.80,

    0.90,

]


# =============================================================================
# 8. Check required files
# =============================================================================

required_files = {

    "selected_sample":
        SELECTED_SAMPLE_FILE,

    "daily":
        DAILY_FILE,

    "weekly":
        WEEKLY_FILE,

    "biweekly":
        BIWEEKLY_FILE,

}


print(
    "\n"
    + "=" * 100
)


print(
    "SOURCE FILE CHECK"
)


print(
    "=" * 100
)


for name, file_path in (
    required_files.items()
):

    print(
        f"{name:20s}",
        file_path.exists(),
        "|",
        file_path,
    )


missing_required = [

    file_path

    for file_path
    in required_files.values()

    if not file_path.exists()

]


if missing_required:

    raise FileNotFoundError(
        "Required files are missing:\n\n"
        +
        "\n".join(
            str(
                file_path
            )
            for file_path
            in missing_required
        )
    )


# =============================================================================
# 9. Load selected sample
# =============================================================================

selected_sample = pd.read_csv(
    SELECTED_SAMPLE_FILE
)


required_sample_columns = [

    "site_id",

    "group",

    "matched_treatment_site_id",

    "control_rank",

]


missing_sample_columns = [

    column

    for column
    in required_sample_columns

    if column not in selected_sample.columns

]


if missing_sample_columns:

    raise ValueError(
        "selected_site_sample.csv is missing:\n"
        +
        str(
            missing_sample_columns
        )
    )


selected_sample[
    "site_id"
] = (
    selected_sample[
        "site_id"
    ]
    .astype(str)
)


selected_sample[
    "group"
] = (
    selected_sample[
        "group"
    ]
    .astype(str)
    .str.lower()
)


selected_sample[
    "matched_treatment_site_id"
] = (
    selected_sample[
        "matched_treatment_site_id"
    ]
    .astype(str)
)


selected_sample[
    "control_rank"
] = pd.to_numeric(
    selected_sample[
        "control_rank"
    ],
    errors=
        "coerce",
)


# =============================================================================
# 10. Validate sample
# =============================================================================

treatment_sample = (
    selected_sample
    .loc[
        selected_sample[
            "group"
        ]
        ==
        "treatment"
    ]
    .copy()
)


counterfactual_sample = (
    selected_sample
    .loc[
        selected_sample[
            "group"
        ]
        ==
        "counterfactual"
    ]
    .copy()
)


controls_per_treatment = (
    counterfactual_sample
    .groupby(
        "matched_treatment_site_id"
    )[
        "site_id"
    ]
    .nunique()
    .rename(
        "counterfactual_count"
    )
    .reset_index()
)


sample_validation = pd.DataFrame(
    {

        "matched_treatment_site_id":
            treatment_sample[
                "site_id"
            ]
            .drop_duplicates()
            .tolist(),

    }
)


sample_validation = (
    sample_validation
    .merge(
        controls_per_treatment,
        on=
            "matched_treatment_site_id",
        how=
            "left",
    )
)


sample_validation[
    "counterfactual_count"
] = (
    sample_validation[
        "counterfactual_count"
    ]
    .fillna(
        0
    )
    .astype(int)
)


sample_validation[
    "expected_counterfactual_count"
] = (
    EXPECTED_CONTROLS_PER_TREATMENT
)


sample_validation[
    "sample_complete"
] = (
    sample_validation[
        "counterfactual_count"
    ]
    ==
    EXPECTED_CONTROLS_PER_TREATMENT
)


sample_validation.to_csv(
    SAMPLE_VALIDATION_OUTPUT,
    index=False,
)


print(
    "\n"
    + "=" * 100
)


print(
    "SAMPLE VALIDATION"
)


print(
    "=" * 100
)


print(
    "\nTreatment sites:",
    treatment_sample[
        "site_id"
    ].nunique()
)


print(
    "Counterfactual sites:",
    counterfactual_sample[
        "site_id"
    ].nunique()
)


print(
    "Total sites:",
    selected_sample[
        "site_id"
    ].nunique()
)


print(
    "\nControls per treatment:"
)


print(
    sample_validation.to_string(
        index=False
    )
)


# =============================================================================
# 11. Quality-column helper
# =============================================================================

def identify_quality_column(
    dataframe,
):

    candidates = [

        "valid_pixel_fraction",

        "valid_pixel_fraction_any_band",

        "valid_pixel_fraction_all_bands",

    ]


    for column in candidates:

        if column in dataframe.columns:

            return column


    raise ValueError(
        "No valid pixel quality variable was found."
    )


# =============================================================================
# 12. Normalize groups
# =============================================================================

def normalize_group(
    value,
):

    value = str(
        value
    ).strip().lower()


    if value in [

        "treatment",

        "treated",

    ]:

        return "treatment"


    if value in [

        "counterfactual",

        "control",

        "controls",

    ]:

        return "counterfactual"


    return value


# =============================================================================
# 13. Normalize sensor
# =============================================================================

def normalize_sensor(
    value,
):

    value = str(
        value
    ).strip().lower()


    if value in [

        "sentinel1",

        "sentinel-1",

        "s1",

    ]:

        return "sentinel1"


    if value in [

        "sentinel2",

        "sentinel-2",

        "s2",

    ]:

        return "sentinel2"


    return value


# =============================================================================
# 14. Before / after helper
# =============================================================================

def assign_period_from_date(
    date,
):

    date = pd.Timestamp(
        date
    )


    if date < HELENE_REFERENCE_DATE:

        return "before"


    return "after"


# =============================================================================
# 15. Standardize primary quality
# =============================================================================

def standardize_quality(
    dataframe,
):

    dataframe = (
        dataframe.copy()
    )


    quality_column = (
        identify_quality_column(
            dataframe
        )
    )


    dataframe[
        "source_quality_column"
    ] = (
        quality_column
    )


    dataframe[
        "quality_fraction"
    ] = pd.to_numeric(
        dataframe[
            quality_column
        ],
        errors=
            "coerce",
    )


    dataframe[
        "quality_percentage"
    ] = (
        dataframe[
            "quality_fraction"
        ]
        *
        100
    )


    dataframe[
        "group"
    ] = (
        dataframe[
            "group"
        ]
        .apply(
            normalize_group
        )
    )


    dataframe[
        "sensor"
    ] = (
        dataframe[
            "sensor"
        ]
        .apply(
            normalize_sensor
        )
    )


    dataframe[
        "site_id"
    ] = (
        dataframe[
            "site_id"
        ]
        .astype(str)
    )


    return dataframe


# =============================================================================
# 16. Attach matching metadata
# =============================================================================

def attach_sample_metadata(
    dataframe,
):

    dataframe = (
        dataframe.copy()
    )


    for column in [

        "matched_treatment_site_id",

        "control_rank",

    ]:

        if column in dataframe.columns:

            dataframe = (
                dataframe
                .drop(
                    columns=
                        column
                )
            )


    metadata = (
        selected_sample[
            [
                "site_id",
                "group",
                "matched_treatment_site_id",
                "control_rank",
            ]
        ]
        .drop_duplicates()
    )


    dataframe = (
        dataframe
        .merge(
            metadata,
            on=[
                "site_id",
                "group",
            ],
            how=
                "left",
        )
    )


    return dataframe


# =============================================================================
# 17. Restrict to selected sample
# =============================================================================

SELECTED_SITE_IDS = set(
    selected_sample[
        "site_id"
    ]
    .astype(str)
)


def restrict_to_selected_sample(
    dataframe,
):

    return (
        dataframe
        .loc[
            dataframe[
                "site_id"
            ]
            .astype(str)
            .isin(
                SELECTED_SITE_IDS
            )
        ]
        .copy()
    )


# =============================================================================
# 18. Load DAILY
# =============================================================================

def load_daily():

    df = pd.read_csv(
        DAILY_FILE
    )


    df = (
        standardize_quality(
            df
        )
    )


    df = (
        restrict_to_selected_sample(
            df
        )
    )


    df = (
        attach_sample_metadata(
            df
        )
    )


    df[
        "dataset"
    ] = (
        "daily"
    )


    df[
        "temporal_resolution"
    ] = (
        "acquisition"
    )


    df[
        "acquisition_date"
    ] = pd.to_datetime(
        df[
            "acquisition_date"
        ]
    )


    df = (
        df
        .loc[
            (
                df[
                    "acquisition_date"
                ]
                >=
                STUDY_START
            )
            &
            (
                df[
                    "acquisition_date"
                ]
                <=
                STUDY_END
            )
        ]
        .copy()
    )


    if "period" not in df.columns:

        df[
            "period"
        ] = (
            df[
                "acquisition_date"
            ]
            .apply(
                assign_period_from_date
            )
        )


    df[
        "time_id"
    ] = (
        df[
            "acquisition_date"
        ]
        .dt.strftime(
            "%Y-%m-%d"
        )
    )


    df[
        "time_start"
    ] = (
        df[
            "acquisition_date"
        ]
    )


    df[
        "time_end"
    ] = (
        df[
            "acquisition_date"
        ]
    )


    # Daily rows correspond to actual acquisitions.
    df[
        "image_available"
    ] = (
        df[
            "quality_fraction"
        ]
        .notna()
        .astype(int)
    )


    return df


# =============================================================================
# 19. Load WEEKLY
# =============================================================================

def load_weekly():

    df = pd.read_csv(
        WEEKLY_FILE
    )


    df = (
        standardize_quality(
            df
        )
    )


    df = (
        restrict_to_selected_sample(
            df
        )
    )


    df = (
        attach_sample_metadata(
            df
        )
    )


    df[
        "dataset"
    ] = (
        "weekly"
    )


    df[
        "temporal_resolution"
    ] = (
        "7_day"
    )


    df[
        "window_start"
    ] = pd.to_datetime(
        df[
            "window_start"
        ]
    )


    df[
        "window_end"
    ] = pd.to_datetime(
        df[
            "window_end"
        ]
    )


    if "week_id" in df.columns:

        df[
            "time_id"
        ] = (
            df[
                "week_id"
            ]
            .astype(str)
        )


    else:

        df[
            "time_id"
        ] = (
            df[
                "period"
            ].astype(str)
            +
            "_W"
            +
            df[
                "week_number"
            ]
            .astype(int)
            .astype(str)
            .str.zfill(
                2
            )
        )


    df[
        "time_start"
    ] = (
        df[
            "window_start"
        ]
    )


    df[
        "time_end"
    ] = (
        df[
            "window_end"
        ]
    )


    if "has_any_acquisition" in df.columns:

        df[
            "image_available"
        ] = pd.to_numeric(
            df[
                "has_any_acquisition"
            ],
            errors=
                "coerce",
        ).fillna(
            0
        ).astype(int)


    else:

        df[
            "image_available"
        ] = (
            df[
                "quality_fraction"
            ]
            .notna()
            .astype(int)
        )


    return df


# =============================================================================
# 20. Load BIWEEKLY
# =============================================================================

def load_biweekly():

    df = pd.read_csv(
        BIWEEKLY_FILE
    )


    df = (
        standardize_quality(
            df
        )
    )


    df = (
        restrict_to_selected_sample(
            df
        )
    )


    df = (
        attach_sample_metadata(
            df
        )
    )


    df[
        "dataset"
    ] = (
        "biweekly"
    )


    df[
        "temporal_resolution"
    ] = (
        "14_day"
    )


    df[
        "period_start"
    ] = pd.to_datetime(
        df[
            "period_start"
        ]
    )


    df[
        "period_end"
    ] = pd.to_datetime(
        df[
            "period_end"
        ]
    )


    if "period_id" in df.columns:

        df[
            "time_id"
        ] = (
            df[
                "period_id"
            ]
            .astype(str)
        )


    else:

        df[
            "time_id"
        ] = (
            df[
                "period"
            ].astype(str)
            +
            "_P"
            +
            df[
                "period_number"
            ]
            .astype(int)
            .astype(str)
            .str.zfill(
                2
            )
        )


    df[
        "time_start"
    ] = (
        df[
            "period_start"
        ]
    )


    df[
        "time_end"
    ] = (
        df[
            "period_end"
        ]
    )


    if "has_data" in df.columns:

        df[
            "image_available"
        ] = pd.to_numeric(
            df[
                "has_data"
            ],
            errors=
                "coerce",
        ).fillna(
            0
        ).astype(int)


    else:

        df[
            "image_available"
        ] = (
            df[
                "quality_fraction"
            ]
            .notna()
            .astype(int)
        )


    return df


# =============================================================================
# 21. Load all three datasets
# =============================================================================

daily = (
    load_daily()
)


weekly = (
    load_weekly()
)


biweekly = (
    load_biweekly()
)


print(
    "\n"
    + "=" * 100
)


print(
    "DATASET ROW COUNTS"
)


print(
    "=" * 100
)


print(
    "\nDaily:",
    len(
        daily
    )
)


print(
    "Weekly:",
    len(
        weekly
    )
)


print(
    "Biweekly:",
    len(
        biweekly
    )
)


# =============================================================================
# 22. Standardized master dataset
# =============================================================================

STANDARD_COLUMNS = [

    "dataset",

    "temporal_resolution",

    "site_id",

    "group",

    "matched_treatment_site_id",

    "control_rank",

    "sensor",

    "period",

    "time_id",

    "time_start",

    "time_end",

    "image_available",

    "quality_fraction",

    "quality_percentage",

    "source_quality_column",

]


standardized_frames = []


for dataframe in [

    daily,

    weekly,

    biweekly,

]:

    for column in STANDARD_COLUMNS:

        if column not in dataframe.columns:

            dataframe[
                column
            ] = (
                np.nan
            )


    standardized_frames.append(
        dataframe[
            STANDARD_COLUMNS
        ]
        .copy()
    )


quality_data = pd.concat(
    standardized_frames,
    ignore_index=True,
)


# =============================================================================
# 23. Images with measurable quality
#
# IMPORTANT:
#
# Missing weekly/biweekly periods are excluded from quality statistics.
#
# They remain in quality_data for availability analysis.
# =============================================================================

quality_images = (
    quality_data
    .loc[
        quality_data[
            "quality_fraction"
        ]
        .notna()
    ]
    .copy()
)


print(
    "\nImages with measurable quality:"
)


print(
    len(
        quality_images
    )
)


# =============================================================================
# 24. Threshold indicators
# =============================================================================

for threshold in QUALITY_THRESHOLDS:

    threshold_pct = int(
        threshold
        *
        100
    )


    quality_images[
        f"quality_ge_{threshold_pct}"
    ] = (
        quality_images[
            "quality_fraction"
        ]
        >=
        threshold
    ).astype(int)


# =============================================================================
# 25. Helper for threshold statistics
# =============================================================================

def add_threshold_statistics(
    summary,
    source,
    group_columns,
    count_prefix,
):

    result = (
        summary.copy()
    )


    for threshold in QUALITY_THRESHOLDS:

        threshold_pct = int(
            threshold
            *
            100
        )


        stats = (
            source
            .groupby(
                group_columns,
                dropna=False,
            )[
                f"quality_ge_{threshold_pct}"
            ]
            .agg(
                [
                    "sum",
                    "mean",
                ]
            )
            .reset_index()
            .rename(
                columns={

                    "sum":
                        (
                            f"{count_prefix}_"
                            f"ge_{threshold_pct}pct"
                        ),

                    "mean":
                        (
                            f"share_{count_prefix}_"
                            f"ge_{threshold_pct}pct"
                        ),

                }
            )
        )


        result = (
            result
            .merge(
                stats,
                on=
                    group_columns,
                how=
                    "left",
            )
        )


        result[
            (
                f"percent_{count_prefix}_"
                f"ge_{threshold_pct}pct"
            )
        ] = (
            result[
                (
                    f"share_{count_prefix}_"
                    f"ge_{threshold_pct}pct"
                )
            ]
            *
            100
        )


    return result


# =============================================================================
# 26. PERIOD DIMENSION
#
# Within each temporal period, compare quality across sites.
# =============================================================================

period_group_columns = [

    "dataset",

    "temporal_resolution",

    "sensor",

    "group",

    "period",

    "time_id",

    "time_start",

    "time_end",

]


period_dimension = (
    quality_images
    .groupby(
        period_group_columns,
        dropna=False,
        as_index=False,
    )
    .agg(

        number_of_images=(
            "site_id",
            "count",
        ),

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        mean_valid_pixel_fraction=(
            "quality_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "quality_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "quality_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "quality_fraction",
            "max",
        ),

        standard_deviation_valid_fraction=(
            "quality_fraction",
            "std",
        ),

    )
)


period_dimension = (
    add_threshold_statistics(

        summary=
            period_dimension,

        source=
            quality_images,

        group_columns=
            period_group_columns,

        count_prefix=
            "images",

    )
)


# =============================================================================
# 27. Convert period fractions to %
# =============================================================================

for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

    (
        "standard_deviation_valid_fraction",
        "standard_deviation_valid_percentage",
    ),

]:

    period_dimension[
        target_column
    ] = (
        period_dimension[
            source_column
        ]
        *
        100
    )


period_dimension = (
    period_dimension
    .sort_values(
        [
            "dataset",
            "sensor",
            "group",
            "period",
            "time_start",
        ]
    )
    .reset_index(
        drop=True
    )
)


period_dimension.to_csv(
    PERIOD_OUTPUT,
    index=False,
)


# =============================================================================
# 28. SITE DIMENSION
#
# Compare image quality across time for each spatial site.
# =============================================================================

site_group_columns = [

    "dataset",

    "temporal_resolution",

    "sensor",

    "group",

    "matched_treatment_site_id",

    "control_rank",

    "period",

    "site_id",

]


site_dimension = (
    quality_images
    .groupby(
        site_group_columns,
        dropna=False,
        as_index=False,
    )
    .agg(

        number_of_images=(
            "time_id",
            "count",
        ),

        number_of_unique_time_periods=(
            "time_id",
            "nunique",
        ),

        mean_valid_pixel_fraction=(
            "quality_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "quality_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "quality_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "quality_fraction",
            "max",
        ),

        standard_deviation_valid_fraction=(
            "quality_fraction",
            "std",
        ),

    )
)


site_dimension = (
    add_threshold_statistics(

        summary=
            site_dimension,

        source=
            quality_images,

        group_columns=
            site_group_columns,

        count_prefix=
            "images",

    )
)


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

    (
        "standard_deviation_valid_fraction",
        "standard_deviation_valid_percentage",
    ),

]:

    site_dimension[
        target_column
    ] = (
        site_dimension[
            source_column
        ]
        *
        100
    )


site_dimension = (
    site_dimension
    .sort_values(
        [
            "dataset",
            "sensor",
            "group",
            "period",
            "matched_treatment_site_id",
            "control_rank",
            "site_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


site_dimension.to_csv(
    SITE_OUTPUT,
    index=False,
)


# =============================================================================
# 29. OVERALL COMPARISON
#
# Main daily vs weekly vs biweekly comparison.
# =============================================================================

overall_group_columns = [

    "dataset",

    "temporal_resolution",

    "sensor",

    "group",

    "period",

]


overall_comparison = (
    quality_images
    .groupby(
        overall_group_columns,
        dropna=False,
        as_index=False,
    )
    .agg(

        number_of_images=(
            "quality_fraction",
            "count",
        ),

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        number_of_temporal_units=(
            "time_id",
            "nunique",
        ),

        mean_valid_pixel_fraction=(
            "quality_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "quality_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "quality_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "quality_fraction",
            "max",
        ),

        standard_deviation_valid_fraction=(
            "quality_fraction",
            "std",
        ),

    )
)


overall_comparison = (
    add_threshold_statistics(

        summary=
            overall_comparison,

        source=
            quality_images,

        group_columns=
            overall_group_columns,

        count_prefix=
            "images",

    )
)


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

    (
        "standard_deviation_valid_fraction",
        "standard_deviation_valid_percentage",
    ),

]:

    overall_comparison[
        target_column
    ] = (
        overall_comparison[
            source_column
        ]
        *
        100
    )


dataset_order = pd.CategoricalDtype(
    categories=[
        "daily",
        "weekly",
        "biweekly",
    ],
    ordered=True,
)


overall_comparison[
    "dataset"
] = (
    overall_comparison[
        "dataset"
    ]
    .astype(
        dataset_order
    )
)


overall_comparison = (
    overall_comparison
    .sort_values(
        [
            "sensor",
            "group",
            "period",
            "dataset",
        ]
    )
    .reset_index(
        drop=True
    )
)


overall_comparison[
    "dataset"
] = (
    overall_comparison[
        "dataset"
    ]
    .astype(str)
)


overall_comparison.to_csv(
    OVERALL_OUTPUT,
    index=False,
)


# =============================================================================
# 30. TEMPORAL AVAILABILITY
#
# Weekly/biweekly have fixed temporal panels.
#
# Daily represents acquisitions and is therefore reported separately.
# =============================================================================

weekly_biweekly_panel = (
    quality_data
    .loc[
        quality_data[
            "dataset"
        ]
        .isin(
            [
                "weekly",
                "biweekly",
            ]
        )
    ]
    .copy()
)


temporal_availability = (
    weekly_biweekly_panel
    .groupby(
        [
            "dataset",
            "temporal_resolution",
            "sensor",
            "group",
            "period",
        ],
        as_index=False,
    )
    .agg(

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        expected_site_periods=(
            "time_id",
            "count",
        ),

        available_site_periods=(
            "image_available",
            "sum",
        ),

    )
)


temporal_availability[
    "missing_site_periods"
] = (
    temporal_availability[
        "expected_site_periods"
    ]
    -
    temporal_availability[
        "available_site_periods"
    ]
)


temporal_availability[
    "availability_rate"
] = (
    temporal_availability[
        "available_site_periods"
    ]
    /
    temporal_availability[
        "expected_site_periods"
    ]
)


temporal_availability[
    "availability_percentage"
] = (
    temporal_availability[
        "availability_rate"
    ]
    *
    100
)


temporal_availability.to_csv(
    AVAILABILITY_OUTPUT,
    index=False,
)


# =============================================================================
# 31. QUALITY DISTRIBUTION
# =============================================================================

def assign_quality_category(
    fraction,
):

    if pd.isna(
        fraction
    ):

        return "missing"


    if fraction >= 0.80:

        return "80-100%"


    if fraction >= 0.60:

        return "60-<80%"


    if fraction >= 0.40:

        return "40-<60%"


    if fraction >= 0.20:

        return "20-<40%"


    if fraction > 0:

        return "0-<20%"


    return "0%"


quality_images[
    "quality_category"
] = (
    quality_images[
        "quality_fraction"
    ]
    .apply(
        assign_quality_category
    )
)


quality_distribution = (
    quality_images
    .groupby(
        [
            "dataset",
            "sensor",
            "group",
            "period",
            "quality_category",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "number_of_images"
        }
    )
)


quality_distribution[
    "total_images"
] = (
    quality_distribution
    .groupby(
        [
            "dataset",
            "sensor",
            "group",
            "period",
        ]
    )[
        "number_of_images"
    ]
    .transform(
        "sum"
    )
)


quality_distribution[
    "percent_images"
] = (
    quality_distribution[
        "number_of_images"
    ]
    /
    quality_distribution[
        "total_images"
    ]
    *
    100
)


quality_distribution.to_csv(
    DISTRIBUTION_OUTPUT,
    index=False,
)


# =============================================================================
# 32. Sentinel-2 matrix helper
#
# Rows    = dataset × temporal period
# Columns = sites
# Values  = valid-pixel percentage
# =============================================================================

def create_quality_matrix(
    dataframe,
    group_name,
    period_name,
):

    subset = (
        dataframe
        .loc[
            (
                dataframe[
                    "sensor"
                ]
                ==
                "sentinel2"
            )
            &
            (
                dataframe[
                    "group"
                ]
                ==
                group_name
            )
            &
            (
                dataframe[
                    "period"
                ]
                ==
                period_name
            )
        ]
        .copy()
    )


    if subset.empty:

        return pd.DataFrame()


    matrix = (
        subset
        .pivot_table(

            index=[
                "dataset",
                "time_id",
                "time_start",
                "time_end",
            ],

            columns=
                "site_id",

            values=
                "quality_percentage",

            aggfunc=
                "mean",

        )
        .reset_index()
    )


    return matrix


# =============================================================================
# 33. Sentinel-2 matrices
# =============================================================================

s2_treatment_before_matrix = (
    create_quality_matrix(
        quality_images,
        "treatment",
        "before",
    )
)


s2_treatment_after_matrix = (
    create_quality_matrix(
        quality_images,
        "treatment",
        "after",
    )
)


s2_counterfactual_before_matrix = (
    create_quality_matrix(
        quality_images,
        "counterfactual",
        "before",
    )
)


s2_counterfactual_after_matrix = (
    create_quality_matrix(
        quality_images,
        "counterfactual",
        "after",
    )
)


matrix_outputs = [

    (
        s2_treatment_before_matrix,
        S2_TREATMENT_BEFORE_MATRIX_OUTPUT,
    ),

    (
        s2_treatment_after_matrix,
        S2_TREATMENT_AFTER_MATRIX_OUTPUT,
    ),

    (
        s2_counterfactual_before_matrix,
        S2_COUNTERFACTUAL_BEFORE_MATRIX_OUTPUT,
    ),

    (
        s2_counterfactual_after_matrix,
        S2_COUNTERFACTUAL_AFTER_MATRIX_OUTPUT,
    ),

]


for matrix, output_file in (
    matrix_outputs
):

    if not matrix.empty:

        matrix.to_csv(
            output_file,
            index=False,
        )


# =============================================================================
# 34. DATASET INVENTORY
#
# Uses complete data including missing scheduled periods.
# =============================================================================

dataset_inventory = (
    quality_data
    .groupby(
        [
            "dataset",
            "sensor",
            "group",
            "period",
        ],
        as_index=False,
    )
    .agg(

        total_rows=(
            "site_id",
            "count",
        ),

        unique_sites=(
            "site_id",
            "nunique",
        ),

        temporal_units=(
            "time_id",
            "nunique",
        ),

        available_images=(
            "image_available",
            "sum",
        ),

        images_with_quality=(
            "quality_fraction",
            lambda x:
                int(
                    x.notna().sum()
                ),
        ),

        missing_quality_rows=(
            "quality_fraction",
            lambda x:
                int(
                    x.isna().sum()
                ),
        ),

    )
)


dataset_inventory.to_csv(
    DATASET_INVENTORY_OUTPUT,
    index=False,
)


# =============================================================================
# 35. Dataset-specific tables
# =============================================================================

daily_quality_table = (
    quality_images
    .loc[
        quality_images[
            "dataset"
        ]
        ==
        "daily"
    ]
    .copy()
)


weekly_quality_table = (
    quality_data
    .loc[
        quality_data[
            "dataset"
        ]
        ==
        "weekly"
    ]
    .copy()
)


biweekly_quality_table = (
    quality_data
    .loc[
        quality_data[
            "dataset"
        ]
        ==
        "biweekly"
    ]
    .copy()
)


# =============================================================================
# 36. Dataset-specific summaries
# =============================================================================

daily_summary = (
    overall_comparison
    .loc[
        overall_comparison[
            "dataset"
        ]
        ==
        "daily"
    ]
    .copy()
)


weekly_summary = (
    overall_comparison
    .loc[
        overall_comparison[
            "dataset"
        ]
        ==
        "weekly"
    ]
    .copy()
)


biweekly_summary = (
    overall_comparison
    .loc[
        overall_comparison[
            "dataset"
        ]
        ==
        "biweekly"
    ]
    .copy()
)


# =============================================================================
# 37. Definitions
# =============================================================================

quality_definitions = pd.DataFrame(
    {

        "variable": [

            "quality_fraction",

            "quality_percentage",

            "image_available",

            "mean_valid_pixel_percentage",

            "median_valid_pixel_percentage",

            "percent_images_ge_80pct",

            "availability_percentage",

            "matched_treatment_site_id",

            "control_rank",

        ],

        "meaning": [

            (
                "Primary image-quality metric. Fraction of spatial pixels "
                "with at least one finite output-band value."
            ),

            (
                "quality_fraction multiplied by 100."
            ),

            (
                "1 when an image exists for the site-period; 0 when the "
                "scheduled weekly/biweekly period has no image."
            ),

            (
                "Mean valid-pixel percentage among images that exist."
            ),

            (
                "Median valid-pixel percentage among images that exist."
            ),

            (
                "Percentage of available images with >=80% valid pixels."
            ),

            (
                "Percentage of scheduled weekly/biweekly site-periods "
                "for which imagery is available."
            ),

            (
                "Treatment site associated with this spatial unit."
            ),

            (
                "Counterfactual rank within the matched set for its "
                "treatment site."
            ),

        ],

    }
)


# =============================================================================
# 38. DAILY Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        DAILY_EXCEL_OUTPUT,
        engine=
            "openpyxl",
    ) as writer:

        daily_quality_table.to_excel(
            writer,
            sheet_name=
                "daily_images",
            index=False,
        )


        daily_summary.to_excel(
            writer,
            sheet_name=
                "summary",
            index=False,
        )


        site_dimension.loc[
            site_dimension[
                "dataset"
            ]
            ==
            "daily"
        ].to_excel(
            writer,
            sheet_name=
                "site_dimension",
            index=False,
        )


        period_dimension.loc[
            period_dimension[
                "dataset"
            ]
            ==
            "daily"
        ].to_excel(
            writer,
            sheet_name=
                "date_dimension",
            index=False,
        )


        selected_sample.to_excel(
            writer,
            sheet_name=
                "selected_sample",
            index=False,
        )


        quality_definitions.to_excel(
            writer,
            sheet_name=
                "definitions",
            index=False,
        )


    print(
        "\nDaily quality Excel saved:"
    )


    print(
        DAILY_EXCEL_OUTPUT
    )


except ModuleNotFoundError:

    print(
        "\nopenpyxl is not installed."
    )


# =============================================================================
# 39. WEEKLY Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        WEEKLY_EXCEL_OUTPUT,
        engine=
            "openpyxl",
    ) as writer:

        weekly_quality_table.to_excel(
            writer,
            sheet_name=
                "weekly_images",
            index=False,
        )


        weekly_summary.to_excel(
            writer,
            sheet_name=
                "summary",
            index=False,
        )


        temporal_availability.loc[
            temporal_availability[
                "dataset"
            ]
            ==
            "weekly"
        ].to_excel(
            writer,
            sheet_name=
                "availability",
            index=False,
        )


        period_dimension.loc[
            period_dimension[
                "dataset"
            ]
            ==
            "weekly"
        ].to_excel(
            writer,
            sheet_name=
                "period_dimension",
            index=False,
        )


        site_dimension.loc[
            site_dimension[
                "dataset"
            ]
            ==
            "weekly"
        ].to_excel(
            writer,
            sheet_name=
                "site_dimension",
            index=False,
        )


        selected_sample.to_excel(
            writer,
            sheet_name=
                "selected_sample",
            index=False,
        )


        quality_definitions.to_excel(
            writer,
            sheet_name=
                "definitions",
            index=False,
        )


    print(
        "\nWeekly quality Excel saved:"
    )


    print(
        WEEKLY_EXCEL_OUTPUT
    )


except ModuleNotFoundError:

    pass


# =============================================================================
# 40. BIWEEKLY Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        BIWEEKLY_EXCEL_OUTPUT,
        engine=
            "openpyxl",
    ) as writer:

        biweekly_quality_table.to_excel(
            writer,
            sheet_name=
                "biweekly_images",
            index=False,
        )


        biweekly_summary.to_excel(
            writer,
            sheet_name=
                "summary",
            index=False,
        )


        temporal_availability.loc[
            temporal_availability[
                "dataset"
            ]
            ==
            "biweekly"
        ].to_excel(
            writer,
            sheet_name=
                "availability",
            index=False,
        )


        period_dimension.loc[
            period_dimension[
                "dataset"
            ]
            ==
            "biweekly"
        ].to_excel(
            writer,
            sheet_name=
                "period_dimension",
            index=False,
        )


        site_dimension.loc[
            site_dimension[
                "dataset"
            ]
            ==
            "biweekly"
        ].to_excel(
            writer,
            sheet_name=
                "site_dimension",
            index=False,
        )


        selected_sample.to_excel(
            writer,
            sheet_name=
                "selected_sample",
            index=False,
        )


        quality_definitions.to_excel(
            writer,
            sheet_name=
                "definitions",
            index=False,
        )


    print(
        "\nBiweekly quality Excel saved:"
    )


    print(
        BIWEEKLY_EXCEL_OUTPUT
    )


except ModuleNotFoundError:

    pass


# =============================================================================
# 41. MAIN COMPARISON Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        COMPARISON_EXCEL_OUTPUT,
        engine=
            "openpyxl",
    ) as writer:

        overall_comparison.to_excel(
            writer,
            sheet_name=
                "overall_comparison",
            index=False,
        )


        temporal_availability.to_excel(
            writer,
            sheet_name=
                "availability",
            index=False,
        )


        period_dimension.to_excel(
            writer,
            sheet_name=
                "period_dimension",
            index=False,
        )


        site_dimension.to_excel(
            writer,
            sheet_name=
                "site_dimension",
            index=False,
        )


        quality_distribution.to_excel(
            writer,
            sheet_name=
                "quality_distribution",
            index=False,
        )


        dataset_inventory.to_excel(
            writer,
            sheet_name=
                "dataset_inventory",
            index=False,
        )


        selected_sample.to_excel(
            writer,
            sheet_name=
                "selected_sample",
            index=False,
        )


        sample_validation.to_excel(
            writer,
            sheet_name=
                "sample_validation",
            index=False,
        )


        quality_definitions.to_excel(
            writer,
            sheet_name=
                "definitions",
            index=False,
        )


        if not s2_treatment_before_matrix.empty:

            s2_treatment_before_matrix.to_excel(
                writer,
                sheet_name=
                    "S2_treat_before",
                index=False,
            )


        if not s2_treatment_after_matrix.empty:

            s2_treatment_after_matrix.to_excel(
                writer,
                sheet_name=
                    "S2_treat_after",
                index=False,
            )


        if not s2_counterfactual_before_matrix.empty:

            s2_counterfactual_before_matrix.to_excel(
                writer,
                sheet_name=
                    "S2_control_before",
                index=False,
            )


        if not s2_counterfactual_after_matrix.empty:

            s2_counterfactual_after_matrix.to_excel(
                writer,
                sheet_name=
                    "S2_control_after",
                index=False,
            )


    print(
        "\nComparison Excel saved:"
    )


    print(
        COMPARISON_EXCEL_OUTPUT
    )


except ModuleNotFoundError:

    print(
        "\nopenpyxl is not installed."
    )


    print(
        "Install with:"
    )


    print(
        "%pip install openpyxl"
    )


    print(
        "\nCSV outputs were still generated."
    )


# =============================================================================
# 42. Print main Sentinel-2 comparison
# =============================================================================

print(
    "\n"
    + "=" * 120
)


print(
    "SENTINEL-2 IMAGE QUALITY: DAILY vs WEEKLY vs BIWEEKLY"
)


print(
    "=" * 120
)


s2_comparison = (
    overall_comparison
    .loc[
        overall_comparison[
            "sensor"
        ]
        ==
        "sentinel2"
    ]
    .copy()
)


columns_to_show = [

    "dataset",

    "group",

    "period",

    "number_of_images",

    "number_of_sites",

    "mean_valid_pixel_percentage",

    "median_valid_pixel_percentage",

    "min_valid_pixel_percentage",

    "max_valid_pixel_percentage",

    "percent_images_ge_40pct",

    "percent_images_ge_60pct",

    "percent_images_ge_80pct",

    "percent_images_ge_90pct",

]


print(
    s2_comparison[
        columns_to_show
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 43. Print Sentinel-1 comparison
# =============================================================================

print(
    "\n"
    + "=" * 120
)


print(
    "SENTINEL-1 IMAGE QUALITY: DAILY vs WEEKLY vs BIWEEKLY"
)


print(
    "=" * 120
)


s1_comparison = (
    overall_comparison
    .loc[
        overall_comparison[
            "sensor"
        ]
        ==
        "sentinel1"
    ]
    .copy()
)


print(
    s1_comparison[
        columns_to_show
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 44. Print temporal availability
# =============================================================================

print(
    "\n"
    + "=" * 120
)


print(
    "WEEKLY / BIWEEKLY TEMPORAL AVAILABILITY"
)


print(
    "=" * 120
)


print(
    temporal_availability[
        [
            "dataset",
            "sensor",
            "group",
            "period",
            "number_of_sites",
            "expected_site_periods",
            "available_site_periods",
            "missing_site_periods",
            "availability_percentage",
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 45. Site coverage check
# =============================================================================

coverage_check = (
    quality_data
    .groupby(
        [
            "dataset",
            "sensor",
            "group",
        ],
        as_index=False,
    )
    .agg(

        unique_sites=(
            "site_id",
            "nunique",
        ),

        rows=(
            "site_id",
            "count",
        ),

    )
)


print(
    "\n"
    + "=" * 120
)


print(
    "SITE COVERAGE CHECK"
)


print(
    "=" * 120
)


print(
    coverage_check.to_string(
        index=False
    )
)


# =============================================================================
# 46. Explicit expected sample check
# =============================================================================

print(
    "\nExpected treatment sites:",
    EXPECTED_TREATMENT_SITES
)


print(
    "Expected counterfactual sites:",
    EXPECTED_COUNTERFACTUAL_SITES
)


print(
    "Expected total sites:",
    EXPECTED_TOTAL_SITES
)


# =============================================================================
# 47. Final summary
# =============================================================================

print(
    "\n"
    + "=" * 120
)


print(
    "NOTEBOOK 13 COMPLETE"
)


print(
    "=" * 120
)


print(
    "\nQuality folder:"
)


print(
    QUALITY_DIR
)


print(
    "\nDaily quality Excel:"
)


print(
    DAILY_EXCEL_OUTPUT
)


print(
    "\nWeekly quality Excel:"
)


print(
    WEEKLY_EXCEL_OUTPUT
)


print(
    "\nBiweekly quality Excel:"
)


print(
    BIWEEKLY_EXCEL_OUTPUT
)


print(
    "\nMain comparison Excel:"
)


print(
    COMPARISON_EXCEL_OUTPUT
)


print(
    "\nMain comparison CSV:"
)


print(
    OVERALL_OUTPUT
)


print(
    "\nTemporal availability CSV:"
)


print(
    AVAILABILITY_OUTPUT
)


print(
    "\nPeriod dimension:"
)


print(
    PERIOD_OUTPUT
)


print(
    "\nSite dimension:"
)


print(
    SITE_OUTPUT
)


print(
    "\nDataset inventory:"
)


print(
    DATASET_INVENTORY_OUTPUT
)


print(
    "\nSample validation:"
)


print(
    SAMPLE_VALIDATION_OUTPUT
)


print(
    "\nNotebook completed successfully."
)